# 05 — Composición interna de los quintiles por trimestre y por mandato

Un quintil es, por construcción, el 20% de la distribución de ingreso per cápita de
**cada trimestre** — su tamaño no cambia nunca. Lo que sí puede cambiar es **qué tan mal
(o bien) vive la gente adentro de cada quintil**, y **cuánto se separa el Q1 del resto**:

- si casi toda la sociedad tiene déficit, Q1 y Q2 se parecen entre sí → déficit
  **extendido** (el problema es de nivel general, no de un grupo puntual);
- si la sociedad mejora en conjunto, el Q1 puede seguir peor que el resto pero menos
  lejos → déficit **concentrado** en el Q1, con nivel más bajo.

Este notebook mide esas dos lecturas (nivel y brecha/sobrerrepresentación) para 12
indicadores de condiciones de vida del hogar, trimestre a trimestre y agregado por
mandato presidencial. **Es transversal**: cada trimestre se mide por separado, no usa el
panel de hogares ni construye ninguna etiqueta de deterioro, y **no atribuye causas a los
gobiernos** — la sección 5 es un test descriptivo de si la composición del Q1 difiere
entre mandatos, no una estimación de impacto causal.

## Cautelas

- **El quiebre de régimen de ingresos de 2016 coincide con el cambio de gobierno**
  (CFK II → Macri): desde 2016 existe `PONDIH`/`DECCFR=12` para separar a quien no
  declara ingreso (D3); antes no. Cualquier diferencia que aparezca justo en ese salto
  puede ser método de medición, no cambio real — no se puede separar una cosa de la otra
  con lo disponible en este repo.
- **`DECCFR` está "a confirmar"**: no lo cubre el codebook oficial (`tipado_nucleo.py`,
  `ESQUEMA_NUCLEO`) — el decil/quintil que arma este notebook es el patrón empírico
  documentado en el notebook 01b y D3, no una definición citada por INDEC.
- **Los quintiles son posiciones relativas** (ranking dentro de cada trimestre, no
  niveles monetarios): no dependen de deflactar con el IPC intervenido de 2007-2015.
- **2020T2 fue un operativo telefónico** (pandemia), no presencial — no comparable sin
  salvedad con el resto de la serie; queda marcado en los gráficos de la sección 2, no
  excluido.
- **`préstamos_o_venta_pertenencias` (V14/V15/V17=1)** es una combinación nueva para este
  notebook, distinta de `recurre_desahorro` (V13|V14|V17) y `recurre_endeudamiento_formal`
  (V15|V16) del notebook 03 — no reemplaza esa lógica, es una tercera agrupación ad-hoc
  pedida para este análisis (préstamo informal + préstamo formal + venta de pertenencias,
  sin el desahorro de V13 ni las compras en cuotas de V16).
- **NA real, no `False` silencioso**: a diferencia de los componentes equivalentes del
  notebook 03 (que convierten un insumo faltante en `False` por cómo compara pandas
  `NaN == valor`), acá cada indicador usa lógica de Kleene (`pandas` `"boolean"` nullable)
  — si falta el insumo, el indicador queda `NA` y ese hogar no entra al denominador de
  *ese* indicador puntual, en vez de contarse como "sin déficit".
- **`tenencia_precaria` (II7∈{3,5,6,7}, inquilino + ocupante en relación de dependencia +
  ocupante gratuito + ocupante de hecho) no resultó un indicador de pobreza limpio** —
  incluir "inquilino" (código 3) mezcla alquiler de mercado (más común en hogares de
  ingresos medios/altos en zonas urbanas) con tenencia informal; se ve en los resultados
  de la sección 3/6, se deja documentado ahí en vez de ocultarlo.

Lee `data/02_nucleo/{hogar,individual}` y `data/01_armonizado/individual` (solo
`NIVEL_ED`), los trimestres de `TRIMESTRES` (`None` = los 86 publicados). No escribe
ninguna capa `data/0N_*` -- solo CSV de resumen en `data/meta/` y, si `GUARDAR_FIGURAS`,
PNG en `figuras/05_composicion_quintiles/` (gitignored).

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.formula.api as smf


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

from armonizacion import trimestres_publicados, es_historico
import panel
import pipeline_meta as pm

TRIMESTRES: list[tuple[int, int]] | None = None  # subconjunto opcional; None = todos
GUARDAR_FIGURAS: bool = True  # guarda PNG en figuras/05_composicion_quintiles/ (gitignored)

NUCLEO = REPO / "data" / "02_nucleo"
ARMONIZADO = REPO / "data" / "01_armonizado"
META = REPO / "data" / "meta"
FIGURAS = REPO / "figuras" / "05_composicion_quintiles"
if GUARDAR_FIGURAS:
    FIGURAS.mkdir(parents=True, exist_ok=True)

TODOS_LOS_TRIMESTRES = trimestres_publicados()
trimestres_a_procesar = TRIMESTRES if TRIMESTRES is not None else TODOS_LOS_TRIMESTRES
ES_CORRIDA_PARCIAL = TRIMESTRES is not None
SUFIJO = "_parcial" if ES_CORRIDA_PARCIAL else ""
COMMIT_GIT = pm.obtener_commit_git(REPO)
t0_proceso = time.time()
print(f"{len(trimestres_a_procesar)} trimestres a procesar" +
      (" -- CORRIDA PARCIAL (TRIMESTRES acotado)" if ES_CORRIDA_PARCIAL else ""))

## Carga de datos y funciones de categorización

Mismo patrón de `cargar_hogar`/`cargar_individual` que los notebooks 03/04 (`NIVEL_ED`
viene de `data/01_armonizado/individual/`, el resto de `data/02_nucleo/`). Las funciones
de quintil (`calcular_grupo_ingreso`), situación laboral del hogar
(`calcular_situacion_laboral`) y bucket educativo del jefe/a (`bucket_nivel_ed`) están
**duplicadas tal cual del notebook 03** (mismo criterio que usó el notebook 04: no hay
módulo compartido en `src/` para estas reglas, y la regla de cambios aditivos impide
tocar 03 para extraerlas). `calcular_info_jefe` se extiende con `CH06` (edad), que la
versión de 03/04 no necesitaba.

In [ ]:
COLUMNAS_HOGAR = [
    "CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "AGLOMERADO", "DECCFR", "PONDIH",
    "II1", "II2", "II5", "II5_1", "II7",
    "IV1", "IV3", "IV8", "IV10", "IV12_1", "IV12_2", "IV12_3",
    "V14", "V15", "V17",
]
COLUMNAS_INDIVIDUAL = [
    "CODUSU", "NRO_HOGAR", "COMPONENTE", "ANO4", "TRIMESTRE",
    "CH03", "CH04", "CH06", "ESTADO", "CAT_OCUP", "PP07H", "PONDERA",
]


def cargar_hogar(anio: int, trimestre: int) -> pd.DataFrame | None:
    p = pm.ruta_particion(NUCLEO, "hogar", anio, trimestre)
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    return df[[c for c in COLUMNAS_HOGAR if c in df.columns]].copy()


def cargar_individual(anio: int, trimestre: int) -> pd.DataFrame | None:
    p = pm.ruta_particion(NUCLEO, "individual", anio, trimestre)
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    ind = df[[c for c in COLUMNAS_INDIVIDUAL if c in df.columns]].copy()
    pa = pm.ruta_particion(ARMONIZADO, "individual", anio, trimestre)
    if pa.exists():
        arm = pd.read_parquet(pa, columns=["CODUSU", "NRO_HOGAR", "COMPONENTE", "NIVEL_ED"])
        arm["NIVEL_ED"] = pd.to_numeric(arm["NIVEL_ED"], errors="coerce").astype("Int8")
        ind = ind.merge(arm, on=["CODUSU", "NRO_HOGAR", "COMPONENTE"], how="left")
    else:
        ind["NIVEL_ED"] = pd.array([pd.NA] * len(ind), dtype="Int8")
    return ind


def calcular_grupo_ingreso(hogar: pd.DataFrame) -> pd.Series:
    '''Duplicada tal cual de 03_exploracion_descriptiva.ipynb (celda 3).'''
    deccfr = pd.to_numeric(hogar["DECCFR"], errors="coerce")
    grupo = pd.Series(pd.array([pd.NA] * len(hogar), dtype="object"), index=hogar.index)
    valido = deccfr.between(1, 10)
    grupo[valido] = "Q" + (((deccfr[valido].astype("Int64") + 1) // 2)).astype(str)
    grupo[deccfr == 0] = "decil_0"
    grupo[deccfr == 12] = "ingreso_no_declarado"
    if "PONDIH" in hogar.columns:
        pondih = pd.to_numeric(hogar["PONDIH"], errors="coerce")
        grupo[(pondih == 0) & grupo.isna()] = "ingreso_no_declarado"
    return grupo


ORDEN_QUINTILES = ["Q1", "Q2", "Q3", "Q4", "Q5"]
ORDEN_MANDATOS = [nombre for (_, nombre) in panel.MANDATO_PRESIDENCIAL]


def calcular_info_jefe(individual: pd.DataFrame) -> pd.DataFrame:
    '''Extiende la versión de 03/04 con `edad_jefe` (CH06), necesaria para el
    indicador 'sin ocupados con jefe/a menor de 65'.'''
    jefe = individual[individual["CH03"] == 1]
    conteo = jefe.groupby(["CODUSU", "NRO_HOGAR"]).size()
    jefe_unico = jefe.drop_duplicates(["CODUSU", "NRO_HOGAR"]).set_index(["CODUSU", "NRO_HOGAR"])
    resultado = pd.DataFrame({
        "pondera_jefe": jefe_unico["PONDERA"],
        "nivel_ed_jefe": jefe_unico["NIVEL_ED"],
        "edad_jefe": pd.to_numeric(jefe_unico["CH06"], errors="coerce"),
    }).reset_index()
    resultado = resultado.merge(conteo.rename("n_jefe").reset_index(), on=["CODUSU", "NRO_HOGAR"], how="left")
    resultado["jefe_ambiguo"] = resultado["n_jefe"].fillna(0).ne(1)
    resultado = resultado.drop(columns=["n_jefe"])
    resultado.loc[resultado["jefe_ambiguo"], ["nivel_ed_jefe", "edad_jefe"]] = pd.NA
    return resultado


def bucket_nivel_ed(nivel_ed: pd.Series) -> pd.Series:
    '''Duplicada tal cual de 03_exploracion_descriptiva.ipynb.'''
    n = pd.to_numeric(nivel_ed, errors="coerce")
    bucket = pd.Series(pd.array([pd.NA] * len(n), dtype="object"), index=n.index)
    bucket[n.isin([1, 2, 3, 7])] = "secundario incompleto o menos"
    bucket[n.isin([4, 5, 6])] = "secundario completo o más"
    return bucket


def calcular_situacion_laboral(individual: pd.DataFrame) -> pd.DataFrame:
    '''Duplicada tal cual de 03_exploracion_descriptiva.ipynb: 4 estados por
    prioridad, algun_asalariado_formal > algun_independiente... > solo_informales >
    sin_ocupados.'''
    estado = pd.to_numeric(individual["ESTADO"], errors="coerce")
    cat_ocup = pd.to_numeric(individual["CAT_OCUP"], errors="coerce")
    pp07h = pd.to_numeric(individual["PP07H"], errors="coerce")
    ocupado = estado == 1
    formal = ocupado & (cat_ocup == 3) & (pp07h == 1)
    independiente = ocupado & cat_ocup.isin([1, 2])
    informal_asalariado = ocupado & (cat_ocup == 3) & (pp07h == 2)

    g = pd.DataFrame({"CODUSU": individual["CODUSU"], "NRO_HOGAR": individual["NRO_HOGAR"],
                       "formal": formal, "independiente": independiente, "informal_asalariado": informal_asalariado})
    ag = g.groupby(["CODUSU", "NRO_HOGAR"])[["formal", "independiente", "informal_asalariado"]].any()
    situacion = pd.Series("sin_ocupados", index=ag.index, dtype="object")
    situacion[ag["informal_asalariado"]] = "solo_asalariados_informales"
    situacion[ag["independiente"]] = "algun_independiente_sin_asalariado_formal"
    situacion[ag["formal"]] = "algun_asalariado_formal"
    return situacion.rename("situacion_laboral").reset_index()

## Los 12 indicadores (hogar, 0/1/NA)

**Vivienda** (mismas reglas que `calcular_componentes_ancla` del notebook 03, confirmadas
contra el codebook p.7-9):
1. `hacinamiento_critico` = miembros/II1 > 3
2. `materiales_deficitarios` = IV1∈{3,4,5} o IV3=3
3. `sanitario_deficitario` = IV8=2 o IV10=3
4. `entorno_deficitario` = IV12_1=1 o IV12_2=1 o IV12_3=1
5. `tenencia_precaria` **[nuevo]** = II7∈{3,5,6,7} (inquilino/arrendatario, ocupante en
   relación de dependencia, ocupante gratuito con permiso, ocupante de hecho sin
   permiso) -- códigos confirmados contra el codebook p.9

**Trabajo** (derivados de `calcular_situacion_laboral` + ESTADO/edad del jefe/a):
6. `ningun_asalariado_formal` = situación laboral del hogar != "algun_asalariado_formal"
7. `solo_asalariados_informales` = situación laboral == "solo_asalariados_informales"
8. `activos_sin_empleo_formal` = algún miembro con ESTADO∈{1,2} (ocupado/desocupado) y
   `ningun_asalariado_formal`
9. `algun_desocupado` = algún miembro con ESTADO=2
10. `sin_ocupados_jefe_menor_65` = situación laboral == "sin_ocupados" y edad del jefe/a
    < 65 (evita contar hogares de jubilados como precariedad laboral)

**Educación y subsistencia**:
11. `jefe_sin_secundario` = NIVEL_ED del jefe/a en {1,2,3,7} (bucket `secundario
    incompleto o menos` de `bucket_nivel_ed`)
12. `prestamos_o_venta_pertenencias` = V14=1 o V15=1 o V17=1 -- códigos confirmados
    contra el codebook p.11-12 (ver cautela de la celda 0)

Cada indicador se calcula con lógica de Kleene (`dtype="boolean"` de pandas): un insumo
faltante deja el resultado en `pd.NA`, nunca en `False`. `to_bool` hace la comparación
nullable de una variable; `kleene_any_hogar` agrega a nivel hogar un indicador de
individual (8, 9) con el mismo criterio: `True` si algún miembro cumple, `NA` si nadie
cumple pero a todos les falta el dato, `False` en el resto.

In [ ]:
def to_bool(serie: pd.Series, codigos: list[int]) -> pd.Series:
    '''Nullable boolean: True si el valor numérico está en `codigos`, NA si el valor
    original no es numérico/está vacío, False en el resto -- a diferencia de
    `pd.to_numeric(...) == valor` (usado en el notebook 03), acá un NaN de origen no se
    convierte en False.'''
    num = pd.to_numeric(serie, errors="coerce")
    out = num.isin(codigos).astype("boolean")
    out[num.isna()] = pd.NA
    return out


def kleene_any_hogar(individual: pd.DataFrame, serie_bool: pd.Series) -> pd.Series:
    '''any() Kleene-aware agrupado por (CODUSU, NRO_HOGAR): True si algún miembro
    cumple, NA si nadie cumple pero todos tienen el dato faltante, False en el resto
    (distinto del `.any()` default de pandas, que trata NA como False).'''
    d = pd.DataFrame({"CODUSU": individual["CODUSU"], "NRO_HOGAR": individual["NRO_HOGAR"], "v": serie_bool})
    g = d.groupby(["CODUSU", "NRO_HOGAR"])
    algun_true = (d["v"].fillna(False) == True).groupby([d["CODUSU"], d["NRO_HOGAR"]]).any()
    todos_na = d["v"].isna().groupby([d["CODUSU"], d["NRO_HOGAR"]]).all()
    resultado = pd.Series(False, index=algun_true.index, dtype="boolean")
    resultado[todos_na] = pd.NA
    resultado[algun_true] = True
    return resultado


INDICADORES = [
    "hacinamiento_critico", "materiales_deficitarios", "sanitario_deficitario",
    "entorno_deficitario", "tenencia_precaria",
    "ningun_asalariado_formal", "solo_asalariados_informales", "activos_sin_empleo_formal",
    "algun_desocupado", "sin_ocupados_jefe_menor_65",
    "jefe_sin_secundario", "prestamos_o_venta_pertenencias",
]
GRUPOS_INDICADORES = {
    "vivienda": ["hacinamiento_critico", "materiales_deficitarios", "sanitario_deficitario",
                 "entorno_deficitario", "tenencia_precaria"],
    "trabajo": ["ningun_asalariado_formal", "solo_asalariados_informales", "activos_sin_empleo_formal",
                "algun_desocupado", "sin_ocupados_jefe_menor_65"],
    "educacion_subsistencia": ["jefe_sin_secundario", "prestamos_o_venta_pertenencias"],
}


def calcular_indicadores_hogar(hogar: pd.DataFrame, individual: pd.DataFrame) -> pd.DataFrame:
    '''Devuelve un DataFrame indexado por (CODUSU, NRO_HOGAR) con los 12 indicadores
    en dtype 'boolean' (NA-aware).'''
    idx = hogar.set_index(["CODUSU", "NRO_HOGAR"]).index
    h = hogar.set_index(["CODUSU", "NRO_HOGAR"])
    n_miembros = individual.groupby(["CODUSU", "NRO_HOGAR"]).size().reindex(idx)

    ii1 = pd.to_numeric(h["II1"], errors="coerce")
    hacinamiento = ((n_miembros / ii1.replace(0, np.nan)) > 3).astype("boolean")
    hacinamiento[ii1.isna() | n_miembros.isna()] = pd.NA

    materiales = to_bool(h["IV1"], [3, 4, 5]) | to_bool(h["IV3"], [3])
    sanitario = to_bool(h["IV8"], [2]) | to_bool(h["IV10"], [3])
    entorno = to_bool(h["IV12_1"], [1]) | to_bool(h["IV12_2"], [1]) | to_bool(h["IV12_3"], [1])
    tenencia = to_bool(h["II7"], [3, 5, 6, 7])
    prestamos = to_bool(h["V14"], [1]) | to_bool(h["V15"], [1]) | to_bool(h["V17"], [1])

    situacion = calcular_situacion_laboral(individual).set_index(["CODUSU", "NRO_HOGAR"])["situacion_laboral"].reindex(idx)
    ningun_formal = (situacion != "algun_asalariado_formal").astype("boolean")
    ningun_formal[situacion.isna()] = pd.NA
    solo_informales = (situacion == "solo_asalariados_informales").astype("boolean")
    solo_informales[situacion.isna()] = pd.NA
    sin_ocupados = situacion == "sin_ocupados"

    algun_activo = kleene_any_hogar(individual, to_bool(individual["ESTADO"], [1, 2])).reindex(idx)
    activos_sin_formal = algun_activo & ningun_formal
    algun_desocupado = kleene_any_hogar(individual, to_bool(individual["ESTADO"], [2])).reindex(idx)

    info_jefe = calcular_info_jefe(individual).set_index(["CODUSU", "NRO_HOGAR"]).reindex(idx)
    edad_jefe = pd.to_numeric(info_jefe["edad_jefe"], errors="coerce")
    sin_ocupados_jefe_menor_65 = (sin_ocupados & (edad_jefe < 65)).astype("boolean")
    sin_ocupados_jefe_menor_65[situacion.isna() | edad_jefe.isna()] = pd.NA

    bucket = bucket_nivel_ed(info_jefe["nivel_ed_jefe"])
    jefe_sin_secundario = (bucket == "secundario incompleto o menos").astype("boolean")
    jefe_sin_secundario[bucket.isna()] = pd.NA

    out = pd.DataFrame({
        "hacinamiento_critico": hacinamiento, "materiales_deficitarios": materiales,
        "sanitario_deficitario": sanitario, "entorno_deficitario": entorno,
        "tenencia_precaria": tenencia,
        "ningun_asalariado_formal": ningun_formal, "solo_asalariados_informales": solo_informales,
        "activos_sin_empleo_formal": activos_sin_formal, "algun_desocupado": algun_desocupado,
        "sin_ocupados_jefe_menor_65": sin_ocupados_jefe_menor_65,
        "jefe_sin_secundario": jefe_sin_secundario, "prestamos_o_venta_pertenencias": prestamos,
    }, index=idx)
    return out, info_jefe["pondera_jefe"]

## Paso 1 — indicadores, pesos (D3) y tasas por trimestre

Para cada trimestre: calcula los 12 indicadores por hogar, el quintil (`DECCFR`), y dos
pesos -- `peso_quintil` (`PONDIH` en la era regular, `pondera_jefe` -- `PONDERA`
individual del jefe/a como proxy del peso del hogar, igual criterio que 03/04 -- en la
histórica, D3) usado para las tasas **dentro de cada quintil**, y `peso_total`
(`pondera_jefe` siempre) usado para la tasa **sobre todos los hogares**, incluidos los
que no tienen quintil válido (`decil_0`/`ingreso_no_declarado`). Junto con las tasas
guarda la verificación del 20% (Paso 2 de la sección 1 más abajo).

In [ ]:
filas_tasa = []
filas_pesos = []
n_filas_total = 0

for anio, trimestre in trimestres_a_procesar:
    hogar = cargar_hogar(anio, trimestre)
    individual = cargar_individual(anio, trimestre)
    if hogar is None or individual is None:
        continue

    idx = hogar.set_index(["CODUSU", "NRO_HOGAR"]).index
    indicadores, pondera_jefe = calcular_indicadores_hogar(hogar, individual)
    grupo = calcular_grupo_ingreso(hogar)
    grupo.index = idx
    n_miembros = individual.groupby(["CODUSU", "NRO_HOGAR"]).size().reindex(idx)

    historico = es_historico(anio, trimestre)
    pondih = (pd.to_numeric(hogar.set_index(["CODUSU", "NRO_HOGAR"])["PONDIH"], errors="coerce")
              if "PONDIH" in hogar.columns else pd.Series(np.nan, index=idx))
    peso_total = pondera_jefe.reindex(idx)
    peso_quintil = peso_total if historico else pondih.reindex(idx)

    n_filas_total += len(idx)

    # --- tasas por indicador x (quintil | total) ---
    for col in INDICADORES:
        serie = indicadores[col]
        mask_t = serie.notna() & peso_total.notna()
        tasa_total = (np.average(serie[mask_t].astype(float), weights=peso_total[mask_t])
                      if mask_t.sum() > 0 and peso_total[mask_t].sum() > 0 else np.nan)
        filas_tasa.append({"anio": anio, "trimestre": trimestre, "indicador": col,
                            "quintil": "total", "tasa": tasa_total, "n_hogares": int(mask_t.sum())})
        for q in ORDEN_QUINTILES:
            mask_q = mask_t & (grupo == q) & peso_quintil.notna()
            w = peso_quintil[mask_q]
            tasa_q = (np.average(serie[mask_q].astype(float), weights=w)
                      if mask_q.sum() > 0 and w.sum() > 0 else np.nan)
            filas_tasa.append({"anio": anio, "trimestre": trimestre, "indicador": col,
                                "quintil": q, "tasa": tasa_q, "n_hogares": int(mask_q.sum())})

    # --- verificación del 20% (sección 1) ---
    poblacion_hogares_total = peso_total.fillna(0).sum()
    poblacion_personas_total = (peso_total.fillna(0) * n_miembros.fillna(0)).sum()
    grupo_pesos = grupo.where(grupo.isin(ORDEN_QUINTILES), "sin_quintil")
    for g, idx_g in grupo_pesos.groupby(grupo_pesos).groups.items():
        peso_g = peso_total.reindex(idx_g).fillna(0)
        personas_g = (peso_g * n_miembros.reindex(idx_g).fillna(0)).sum()
        filas_pesos.append({
            "anio": anio, "trimestre": trimestre, "era": "historica" if historico else "regular",
            "quintil": g, "n_hogares": len(idx_g),
            "pct_hogares": round(peso_g.sum() / poblacion_hogares_total * 100, 2) if poblacion_hogares_total else np.nan,
            "pct_personas": round(personas_g / poblacion_personas_total * 100, 2) if poblacion_personas_total else np.nan,
        })

tasa_trimestre = pd.DataFrame(filas_tasa)
pesos_trimestre = pd.DataFrame(filas_pesos)
print(f"{len(tasa_trimestre):,} filas de tasa, {len(pesos_trimestre):,} filas de pesos, "
      f"{n_filas_total:,} hogares procesados en {len(trimestres_a_procesar)} trimestres")

## Sección 1 — Verificación del 20%

`DECCFR` reparte **hogares**, no personas: los hogares pobres son en promedio más
grandes, así que el Q1 pesa más del 20% de las *personas* aunque pese ~20% de los
*hogares*. La tabla de abajo muestra el % ponderado de hogares y de personas por quintil,
separado por era, más el % de hogares sin quintil (`decil_0` + `ingreso_no_declarado`).

In [ ]:
resumen_pesos = (pesos_trimestre.groupby(["era", "quintil"])[["pct_hogares", "pct_personas"]]
                  .mean().round(1)
                  .reindex(pd.MultiIndex.from_product([["historica", "regular"], ORDEN_QUINTILES + ["sin_quintil"]],
                                                       names=["era", "quintil"])))
pesos_trimestre.to_csv(META / f"composicion_quintiles_pesos{SUFIJO}.csv", index=False)
print("--- % promedio de hogares y de personas por quintil y era (tabla a mostrar) ---")
display(resumen_pesos)

## Estilo de gráficos y eje temporal continuo

Paleta ordinal de azules (Q1 el más oscuro, los quintiles son posiciones ordenadas, no
categorías sueltas) + total en gris punteado. Eje x = período continuo (`anio*4 +
trimestre`), para que los trimestres no publicados (2007T3, 2015T3-2016T1, y los
aglomerados puntuales faltantes de 2019T3/2020T3 que no vacían el trimestre entero)
queden como huecos en la línea en vez de unir visualmente períodos no contiguos.

In [ ]:
COLOR_QUINTIL = {"Q1": "#0d366b", "Q2": "#1c5cab", "Q3": "#2a78d6", "Q4": "#5598e7", "Q5": "#86b6ef"}
COLOR_TOTAL = "#52514e"


def periodo(anio: int, trimestre: int) -> int:
    return anio * 4 + trimestre


def periodo_inverso(p: int) -> tuple[int, int]:
    trimestre = (p - 1) % 4 + 1
    anio = (p - 1) // 4
    return anio, trimestre


def segmentos_mandato(periodos: list[int]) -> list[tuple[int, int, str]]:
    '''Tramos contiguos (periodo_desde, periodo_hasta, nombre_mandato) para sombrear
    el fondo -- se calcula sobre el calendario (panel.mandato_presidencial), no sobre
    los datos, así cubre también los huecos sin publicación.'''
    segmentos = []
    inicio = periodos[0]
    actual = panel.mandato_presidencial(*periodo_inverso(periodos[0]))
    for p in periodos[1:]:
        nombre = panel.mandato_presidencial(*periodo_inverso(p))
        if nombre != actual:
            segmentos.append((inicio, p - 1, actual))
            inicio, actual = p, nombre
    segmentos.append((inicio, periodos[-1], actual))
    return segmentos


def aplicar_estilo_eje(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.3, linewidth=0.6)
    ax.set_axisbelow(True)


def sombrear_mandatos(ax, segmentos, con_etiqueta=True, alpha=0.07):
    for i, (p_desde, p_hasta, nombre) in enumerate(segmentos):
        ax.axvspan(p_desde - 0.5, p_hasta + 0.5, color="#888888" if i % 2 else "#dddddd", alpha=alpha, zorder=0)
        if con_etiqueta:
            ax.text((p_desde + p_hasta) / 2, 1.02, nombre, transform=ax.get_xaxis_transform(),
                    ha="center", va="bottom", fontsize=7.5, rotation=0, color="#444444")
    ax.axvline(periodo(2016, 2), color="#b33", linewidth=1, linestyle=":", alpha=0.8, zorder=1)
    marca_2020t2 = periodo(2020, 2)
    ax.axvline(marca_2020t2, color="#999", linewidth=5, alpha=0.18, zorder=0)


def etiquetas_eje_x(ax, periodos):
    anio_min, _ = periodo_inverso(periodos[0])
    anio_max, _ = periodo_inverso(periodos[-1])
    anios_tick = range(anio_min + (1 if periodo_inverso(periodos[0])[1] != 1 else 0), anio_max + 1, 2)
    ticks = [periodo(a, 1) for a in anios_tick]
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(a) for a in anios_tick], rotation=45, ha="right", fontsize=8)

## Sección 2 — Nivel en el tiempo

Small multiples por grupo de indicadores: una línea por quintil + el total punteado,
fondo alternado por mandato (con el nombre arriba), línea vertical roja en 2016T2
(quiebre de régimen de ingresos, D3) y una franja gris ancha en 2020T2 (operativo
telefónico).

In [ ]:
periodos_procesados = sorted({periodo(a, t) for a, t in trimestres_a_procesar})
p_min, p_max = periodos_procesados[0], periodos_procesados[-1]
EJE_COMPLETO = list(range(p_min, p_max + 1))
SEGMENTOS_MANDATO = segmentos_mandato(EJE_COMPLETO)

tasa_pivot = tasa_trimestre.copy()
tasa_pivot["periodo"] = tasa_pivot.apply(lambda r: periodo(int(r["anio"]), int(r["trimestre"])), axis=1)


def graficar_nivel_grupo(nombre_grupo: str, indicadores_grupo: list[str]):
    fig, axes = plt.subplots(len(indicadores_grupo), 1, figsize=(10, 2.6 * len(indicadores_grupo)), sharex=True)
    if len(indicadores_grupo) == 1:
        axes = [axes]
    for ax, indicador in zip(axes, indicadores_grupo):
        sub = tasa_pivot[tasa_pivot["indicador"] == indicador]
        for q in ORDEN_QUINTILES:
            serie = (sub[sub["quintil"] == q].set_index("periodo")["tasa"] * 100).reindex(EJE_COMPLETO)
            ax.plot(EJE_COMPLETO, serie.values, color=COLOR_QUINTIL[q], linewidth=1.8, label=q)
        total = (sub[sub["quintil"] == "total"].set_index("periodo")["tasa"] * 100).reindex(EJE_COMPLETO)
        ax.plot(EJE_COMPLETO, total.values, color=COLOR_TOTAL, linewidth=1.6, linestyle=":", label="Total")
        sombrear_mandatos(ax, SEGMENTOS_MANDATO, con_etiqueta=(ax is axes[0]))
        aplicar_estilo_eje(ax)
        ax.set_ylabel("% hogares")
        ax.text(0.01, 0.97, indicador, transform=ax.transAxes, fontsize=10, fontweight="bold",
                va="top", ha="left", bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=1.5), zorder=5)
        ax.set_xlim(p_min - 1, p_max + 1)
    axes[0].legend(ncols=6, fontsize=8, loc="upper right", frameon=False)
    etiquetas_eje_x(axes[-1], EJE_COMPLETO)
    fig.suptitle(f"Nivel por quintil en el tiempo -- {nombre_grupo}", fontsize=12, y=1.0)
    fig.tight_layout()
    if GUARDAR_FIGURAS:
        fig.savefig(FIGURAS / f"nivel_{nombre_grupo}.png", dpi=150, bbox_inches="tight")
    plt.show()


for nombre_grupo, indicadores_grupo in GRUPOS_INDICADORES.items():
    graficar_nivel_grupo(nombre_grupo, indicadores_grupo)

## Sección 3 — Por mandato

Promedio (y desvío) de los trimestres de cada mandato, por indicador y quintil/total.
`n_trimestres` es la cantidad de trimestres **publicados y con tasa no nula** dentro de
ese mandato (no todos los mandatos tienen la misma cantidad -- Kirchner y Milei están
truncados por los bordes de la serie, 2003T3-2025T4).

In [ ]:
tasa_trimestre["mandato"] = tasa_trimestre.apply(lambda r: panel.mandato_presidencial(int(r["anio"]), int(r["trimestre"])), axis=1)

tabla_mandato = (tasa_trimestre.groupby(["indicador", "mandato", "quintil"])["tasa"]
                  .agg(promedio="mean", desvio="std", n_trimestres="count").reset_index())
tabla_mandato["mandato"] = pd.Categorical(tabla_mandato["mandato"], categories=ORDEN_MANDATOS, ordered=True)
tabla_mandato = tabla_mandato.sort_values(["indicador", "mandato", "quintil"])
tabla_mandato.to_csv(META / f"composicion_quintiles_mandato{SUFIJO}.csv", index=False)

MANDATOS_PRESENTES = [m for m in ORDEN_MANDATOS if m in tasa_trimestre["mandato"].unique()]
print(f"{len(MANDATOS_PRESENTES)} mandatos presentes: {MANDATOS_PRESENTES}")
tabla_mandato.head(8)

In [ ]:
def graficar_por_mandato_grupo(nombre_grupo: str, indicadores_grupo: list[str]):
    fig, axes = plt.subplots(len(indicadores_grupo), 1, figsize=(9, 2.6 * len(indicadores_grupo)), sharex=True)
    if len(indicadores_grupo) == 1:
        axes = [axes]
    xs = list(range(len(MANDATOS_PRESENTES)))
    for ax, indicador in zip(axes, indicadores_grupo):
        sub = tabla_mandato[tabla_mandato["indicador"] == indicador].set_index(["mandato", "quintil"])["promedio"]
        for i, mandato in enumerate(MANDATOS_PRESENTES):
            vals_q = [sub.get((mandato, q), np.nan) * 100 for q in ORDEN_QUINTILES]
            vals_q_validos = [v for v in vals_q if pd.notna(v)]
            if vals_q_validos:
                ax.vlines(i, min(vals_q_validos), max(vals_q_validos), color="#cccccc", linewidth=7, zorder=1)
            for q, v in zip(ORDEN_QUINTILES, vals_q):
                if pd.notna(v):
                    ax.scatter(i, v, color=COLOR_QUINTIL[q], zorder=3, s=45, label=q if i == 0 else None)
            total_v = sub.get((mandato, "total"), np.nan) * 100
            if pd.notna(total_v):
                ax.hlines(total_v, i - 0.22, i + 0.22, color=COLOR_TOTAL, linestyle=":", linewidth=2.2, zorder=2,
                          label="Total" if i == 0 else None)
        aplicar_estilo_eje(ax)
        ax.set_xlim(-0.5, len(xs) - 0.5)
        ax.set_ylabel("% hogares")
        ax.set_title(indicador, fontsize=10, loc="left")
    axes[0].legend(ncols=6, fontsize=8, loc="upper left", frameon=False)
    axes[-1].set_xticks(xs)
    axes[-1].set_xticklabels(MANDATOS_PRESENTES, rotation=30, ha="right", fontsize=8)
    fig.suptitle(f"Promedio por mandato -- {nombre_grupo}", fontsize=12, y=1.0)
    fig.tight_layout()
    if GUARDAR_FIGURAS:
        fig.savefig(FIGURAS / f"mandato_{nombre_grupo}.png", dpi=150, bbox_inches="tight")
    plt.show()


for nombre_grupo, indicadores_grupo in GRUPOS_INDICADORES.items():
    graficar_por_mandato_grupo(nombre_grupo, indicadores_grupo)

## Sección 4 — Brecha y sobrerrepresentación en el tiempo

`brecha_q1_q5` = tasa(Q1) − tasa(Q5) en puntos porcentuales. `sobrerrepresentacion_q1` =
tasa(Q1) / tasa(total) (y lo mismo para Q5) -- **≈1 = déficit extendido** (el Q1 no se
distingue del total), **≫1 = déficit concentrado en el Q1**.

In [ ]:
ancho = tasa_pivot.pivot_table(index=["anio", "trimestre", "indicador"], columns="quintil", values="tasa").reset_index()
ancho["mandato"] = ancho.apply(lambda r: panel.mandato_presidencial(int(r["anio"]), int(r["trimestre"])), axis=1)
ancho["brecha_q1_q5_pp"] = (ancho["Q1"] - ancho["Q5"]) * 100
ancho["sobrerrepresentacion_q1"] = ancho["Q1"] / ancho["total"]
ancho["sobrerrepresentacion_q5"] = ancho["Q5"] / ancho["total"]
brechas = ancho[["anio", "trimestre", "indicador", "mandato", "Q1", "Q5", "total",
                  "brecha_q1_q5_pp", "sobrerrepresentacion_q1", "sobrerrepresentacion_q5"]].copy()
brechas.to_csv(META / f"composicion_quintiles_brechas{SUFIJO}.csv", index=False)
brechas.head(3)

In [ ]:
def graficar_sobrerrepresentacion_grupo(nombre_grupo: str, indicadores_grupo: list[str]):
    fig, axes = plt.subplots(len(indicadores_grupo), 1, figsize=(10, 2.3 * len(indicadores_grupo)), sharex=True)
    if len(indicadores_grupo) == 1:
        axes = [axes]
    for ax, indicador in zip(axes, indicadores_grupo):
        sub = brechas[brechas["indicador"] == indicador].copy()
        sub["periodo"] = sub.apply(lambda r: periodo(int(r["anio"]), int(r["trimestre"])), axis=1)
        sub = sub.set_index("periodo")
        ax.plot(EJE_COMPLETO, sub["sobrerrepresentacion_q1"].reindex(EJE_COMPLETO).values,
                color=COLOR_QUINTIL["Q1"], linewidth=1.8, label="Q1 / total")
        ax.plot(EJE_COMPLETO, sub["sobrerrepresentacion_q5"].reindex(EJE_COMPLETO).values,
                color=COLOR_QUINTIL["Q5"], linewidth=1.8, label="Q5 / total")
        ax.axhline(1.0, color="#999999", linewidth=1, linestyle="--", zorder=1)
        sombrear_mandatos(ax, SEGMENTOS_MANDATO, con_etiqueta=(ax is axes[0]))
        aplicar_estilo_eje(ax)
        ax.set_ylabel("razón vs. total")
        ax.text(0.01, 0.97, indicador, transform=ax.transAxes, fontsize=10, fontweight="bold",
                va="top", ha="left", bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=1.5), zorder=5)
        ax.set_xlim(p_min - 1, p_max + 1)
    axes[0].legend(ncols=3, fontsize=8, loc="upper right", frameon=False)
    etiquetas_eje_x(axes[-1], EJE_COMPLETO)
    fig.suptitle(f"Sobrerrepresentación en el tiempo -- {nombre_grupo}", fontsize=12, y=1.0)
    fig.tight_layout()
    if GUARDAR_FIGURAS:
        fig.savefig(FIGURAS / f"sobrerrepresentacion_{nombre_grupo}.png", dpi=150, bbox_inches="tight")
    plt.show()


for nombre_grupo, indicadores_grupo in GRUPOS_INDICADORES.items():
    graficar_sobrerrepresentacion_grupo(nombre_grupo, indicadores_grupo)

## Sección 5 — Test por mandato

Para `brecha_q1_q5_pp` y `sobrerrepresentacion_q1`, OLS de la serie trimestral sobre
dummies de mandato (`C(mandato)`), errores HAC (Newey-West, 4 rezagos) y test F conjunto
de que las dummies son cero -- equivale a testear si el promedio de la variable difiere
entre mandatos, con una covarianza robusta a la autocorrelación serial de trimestres
consecutivos dentro de un mismo mandato. **Con 6 mandatos esto es un test descriptivo,
no causal** -- no identifica qué política causó qué diferencia, solo si hay diferencia.

También reporta: el mandato con el valor promedio mínimo y máximo, el desvío promedio
*dentro* de cada mandato (cuánto varía la variable entre trimestres de un mismo
gobierno, para comparar contra cuánto varía *entre* mandatos), y el salto entre el
promedio de 2015 (último año de la era histórica, D3) y el de 2016 (primer año de la
era regular) -- para poder decir, cuando la brecha sí difiere entre mandatos, si ese
salto por sí solo explica una parte comparable de la diferencia (en ese caso: "puede ser
método, D3", no necesariamente cambio real).

In [ ]:
def test_mandato(df_indicador: pd.DataFrame, variable: str) -> dict:
    datos = df_indicador[["mandato", variable, "anio"]].dropna(subset=[variable]).copy()
    datos["mandato"] = datos["mandato"].astype(str)
    promedio_por_mandato = datos.groupby("mandato")[variable].mean()
    desvio_por_mandato = datos.groupby("mandato")[variable].std()
    salto_historica_regular = (datos.loc[datos["anio"] == 2016, variable].mean()
                                - datos.loc[datos["anio"] == 2015, variable].mean())

    resultado = {
        "mandato_minimo": promedio_por_mandato.idxmin() if len(promedio_por_mandato) else None,
        "valor_minimo": promedio_por_mandato.min() if len(promedio_por_mandato) else np.nan,
        "mandato_maximo": promedio_por_mandato.idxmax() if len(promedio_por_mandato) else None,
        "valor_maximo": promedio_por_mandato.max() if len(promedio_por_mandato) else np.nan,
        "desvio_promedio_intra_mandato": desvio_por_mandato.mean(),
        "salto_historica_regular": salto_historica_regular,
        "f_stat": np.nan, "f_pvalue": np.nan, "n_obs": len(datos),
    }
    if datos["mandato"].nunique() >= 2 and len(datos) >= 10:
        modelo = smf.ols(f"{variable} ~ C(mandato)", data=datos).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
        dummies = [p for p in modelo.params.index if p != "Intercept"]
        if dummies:
            ft = modelo.f_test(", ".join(f"{p} = 0" for p in dummies))
            resultado["f_stat"] = float(ft.fvalue)
            resultado["f_pvalue"] = float(ft.pvalue)
    return resultado


filas_test = []
for indicador in INDICADORES:
    sub = ancho[ancho["indicador"] == indicador]
    for variable in ("brecha_q1_q5_pp", "sobrerrepresentacion_q1"):
        fila = {"indicador": indicador, "variable": variable}
        fila.update(test_mandato(sub, variable))
        filas_test.append(fila)

tabla_test_mandato = pd.DataFrame(filas_test)
tabla_test_mandato.to_csv(META / f"composicion_quintiles_test_mandato{SUFIJO}.csv", index=False)
print("--- tabla del test por mandato (a mostrar) ---")
display(tabla_test_mandato.round(4))

## Sección 6 — Resumen en texto

Una línea por indicador, generada directamente desde `tabla_mandato` y
`tabla_test_mandato` (no texto escrito a mano). Clasificación de
extendido/moderado/concentrado según la sobrerrepresentación máxima del Q1 entre
mandatos: **< 1.3× extendido, 1.3-2.5× moderado, > 2.5× concentrado** -- son umbrales de
referencia para describir, no un criterio validado en la literatura.

In [ ]:
def clasificar_concentracion(sobrerrep_max: float) -> str:
    if pd.isna(sobrerrep_max):
        return "sin dato"
    if sobrerrep_max < 1.3:
        return "extendido"
    if sobrerrep_max <= 2.5:
        return "moderado"
    return "concentrado"


lineas_resumen = []
for indicador in INDICADORES:
    niveles = tabla_mandato[(tabla_mandato["indicador"] == indicador) & (tabla_mandato["quintil"].isin(["Q1", "Q5", "total"]))]
    rango = niveles.groupby("quintil")["promedio"].agg(["min", "max"]) * 100

    sobrerrep = ancho[ancho["indicador"] == indicador].groupby(
        ancho[ancho["indicador"] == indicador]["mandato"].astype(str))["sobrerrepresentacion_q1"].mean()
    sobrerrep_min, sobrerrep_max = (sobrerrep.min(), sobrerrep.max()) if len(sobrerrep) else (np.nan, np.nan)
    concentracion = clasificar_concentracion(sobrerrep_max)

    fila_test = tabla_test_mandato[(tabla_test_mandato["indicador"] == indicador)
                                    & (tabla_test_mandato["variable"] == "brecha_q1_q5_pp")].iloc[0]
    difiere = pd.notna(fila_test["f_pvalue"]) and fila_test["f_pvalue"] < 0.05

    def fmt_rango(etiqueta):
        if etiqueta not in rango.index:
            return f"{etiqueta}=s/d"
        return f"{etiqueta} {rango.loc[etiqueta, 'min']:.1f}-{rango.loc[etiqueta, 'max']:.1f}%"

    base = (f"**{indicador}**: " + ", ".join(fmt_rango(e) for e in ["Q1", "Q5", "total"])
            + f" entre mandatos; sobrerrepresentación Q1/total {sobrerrep_min:.2f}-{sobrerrep_max:.2f}x "
              f"-> déficit {concentracion}.")

    if difiere:
        rango_brecha = fila_test["valor_maximo"] - fila_test["valor_minimo"]
        salto = fila_test["salto_historica_regular"]
        comparable = (pd.notna(salto) and rango_brecha not in (0, np.nan)
                      and abs(salto) >= 0.4 * abs(rango_brecha))
        base += (f" La brecha Q1-Q5 difiere entre mandatos (F={fila_test['f_stat']:.1f}, "
                 f"p={fila_test['f_pvalue']:.4f}); mínimo en {fila_test['mandato_minimo']}, "
                 f"máximo en {fila_test['mandato_maximo']}.")
        if comparable:
            base += " El salto 2015->2016 es comparable a esa diferencia entre mandatos -- puede ser método, D3."
    else:
        base += (" La composición del Q1 no cambia entre mandatos más que dentro de ellos "
                 f"(F={fila_test['f_stat']:.1f}, p={fila_test['f_pvalue']:.4f})." if pd.notna(fila_test["f_pvalue"])
                 else " No hay suficientes observaciones para el test por mandato.")

    lineas_resumen.append(base)

print("\n\n".join(lineas_resumen))

## Registro de la corrida

In [ ]:
duracion = time.time() - t0_proceso
pm.registrar_corrida(
    META / "corridas.csv", "05_composicion_quintiles",
    {"TRIMESTRES": trimestres_a_procesar if ES_CORRIDA_PARCIAL else None, "GUARDAR_FIGURAS": GUARDAR_FIGURAS},
    len(trimestres_a_procesar), n_filas_total, duracion, COMMIT_GIT,
)
print(f"corrida registrada -- {len(trimestres_a_procesar)} trimestres, {n_filas_total:,} hogares, {duracion:.1f}s")